In [1]:
import hashlib
import json
from datetime import datetime

class RiskGuardSystem:
    def __init__(self):
        self.pending_requests = {} # العمليات المعلقة في انتظار التحقق
        self.audit_log = []        # سجل التدقيق الأمني
        self.last_log_hash = "0"   # بصمة السجل السابق (مفهوم شبيه بسلسلة الكتل لضمان عدم التلاعب)

    def create_transaction(self, maker_name, amount, description):
        """يقوم الصانع (Maker) بإنشاء طلب عملية جديدة"""
        tx_id = f"TXN-{int(datetime.now().timestamp())}"
        
        # تقييم مخاطر تلقائي بناءً على المبلغ
        risk_level = "High (عالي)" if amount > 50000 else "Medium (متوسط)"
        
        transaction = {
            "Transaction_ID": tx_id,
            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Maker": maker_name,
            "Amount": amount,
            "Description": description,
            "Risk_Level": risk_level,
            "Status": "Pending Approval (في انتظار الاعتماد)"
        }
        
        self.pending_requests[tx_id] = transaction
        print(f"\n[+] [Maker] قام الموظف ({maker_name}) بإنشاء عملية برقم: {tx_id}")
        print(f"    ⚠️ تقييم المخاطر التلقائي للعملية: {risk_level}")
        return tx_id

    def verify_transaction(self, tx_id, checker_name, action):
        """يقوم المتحقق (Checker) بمراجعة العملية (اعتماد أو رفض)"""
        if tx_id not in self.pending_requests:
            print("❌ خطأ: رقم العملية غير موجود أو تم معالجته مسبقاً!")
            return

        tx = self.pending_requests[tx_id]

        # حماية أمنية: منع الصانع من أن يكون هو نفسه المتحقق لضمان الفصل بين الواجبات
        if tx["Maker"] == checker_name:
            print(f"🚨 [إنتهاك أمني] لا يمكن للموظف ({checker_name}) اعتماد عملية قام بإنشائها بنفسه!")
            return

        if action.lower() in ['اعتماد', 'approve', 'y']:
            tx["Status"] = "Approved & Executed"
            print(f"✅ [Checker] قام المشرف ({checker_name}) باعتماد وتنفيذ العملية {tx_id} بنجاح.")
        else:
            tx["Status"] = "Rejected"
            print(f"❌ [Checker] قام المشرف ({checker_name}) برفض العملية {tx_id}.")

        tx["Checker"] = checker_name
        tx["Verification_Time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # نقل العملية إلى سجل التدقيق المشفر وتطهير قائمة المعلقات
        self._add_to_immutable_audit_log(tx)
        del self.pending_requests[tx_id]

    def _add_to_immutable_audit_log(self, transaction):
        """إضافة العملية إلى سجل تدقيق مشفر بربط كل سجل بالذي قبله (Chain of Logs)"""
        tx_string = json.dumps(transaction, sort_keys=True)
        
        # دمج بيانات العملية الحالية مع بصمة السجل السابق لمنع تعديل السجلات القديمة
        combined_data = tx_string + self.last_log_hash
        current_hash = hashlib.sha256(combined_data.encode()).hexdigest()
        
        audit_entry = {
            "Data": transaction,
            "Previous_Hash": self.last_log_hash,
            "Current_Hash": current_hash
        }
        
        self.audit_log.append(audit_entry)
        self.last_log_hash = current_hash # تحديث البصمة المرجعية للسجل القادم

    def display_audit_trail(self):
        """عرض سجل التدقيق كاملاً لغايات الـ Compliance والرقابة"""
        print("\n" + "="*30 + " 📊 سجل التدقيق الأمني (Audit Trail) " + "="*30)
        for entry in self.audit_log:
            print(f"ID: {entry['Data']['Transaction_ID']} | الحالة: {entry['Data']['Status']}")
            print(f"الصانع (Maker): {entry['Data']['Maker']} | المتحقق (Checker): {entry['Data'].get('Checker', 'N/A')}")
            print(f"المبلغ: {entry['Data']['Amount']} | المخاطر: {entry['Data']['Risk_Level']}")
            print(f"🔗 الـ Hash الحالي للسجل: {entry['Current_Hash']}")
            print("-" * 80)

if __name__ == "__main__":
    print(" 🛡️ RiskGuard: Maker-Checker & Immutable Audit System 🛡️ ")
    system = RiskGuardSystem()

    # سيناريو تجريبي يحاكي العمل الحقيقي
    # 1. إنشاء عمليات بواسطة Maker
    tx1 = system.create_transaction(maker_name="أحمد", amount=75000, description="شراء خوادم جديدة للموقع")
    tx2 = system.create_transaction(maker_name="سارة", amount=12000, description="تجديد تراخيص البرمجيات")

    # 2. محاولة اختراق الضوابط (أحمد يحاول اعتماد عمليته بنفسه)
    print("\n--- محاولة فحص اختراق الضوابط الأمنية ---")
    system.verify_transaction(tx_id=tx1, checker_name="أحمد", action="اعتماد")

    # 3. الاعتماد الصحيح والمنفصل للواجبات
    print("\n--- معالجة العمليات بشكل صحيح ---")
    system.verify_transaction(tx_id=tx1, checker_name="ريناد", action="اعتماد") # ريناد تدقق على أحمد
    system.verify_transaction(tx_id=tx2, checker_name="أحمد", action="اعتماد")  # أحمد يدقق على سارة

    # 4. طباعة السجل النهائي للامتثال
    system.display_audit_trail()

 🛡️ RiskGuard: Maker-Checker & Immutable Audit System 🛡️ 

[+] [Maker] قام الموظف (أحمد) بإنشاء عملية برقم: TXN-1781523191
    ⚠️ تقييم المخاطر التلقائي للعملية: High (عالي)

[+] [Maker] قام الموظف (سارة) بإنشاء عملية برقم: TXN-1781523191
    ⚠️ تقييم المخاطر التلقائي للعملية: Medium (متوسط)

--- محاولة فحص اختراق الضوابط الأمنية ---
✅ [Checker] قام المشرف (أحمد) باعتماد وتنفيذ العملية TXN-1781523191 بنجاح.

--- معالجة العمليات بشكل صحيح ---
❌ خطأ: رقم العملية غير موجود أو تم معالجته مسبقاً!
❌ خطأ: رقم العملية غير موجود أو تم معالجته مسبقاً!

============================== 📊 سجل التدقيق الأمني (Audit Trail) ==============================
ID: TXN-1781523191 | الحالة: Approved & Executed
الصانع (Maker): سارة | المتحقق (Checker): أحمد
المبلغ: 12000 | المخاطر: Medium (متوسط)
🔗 الـ Hash الحالي للسجل: 011c50e3422697a9f2fbb8bb76aab2a81d91e65afa95801cf4d21fc26935ee5c
--------------------------------------------------------------------------------


In [1]:
import hashlib
import json
import uuid
from datetime import datetime

class RiskGuardSystem:
    def __init__(self):
        # استخدام القواميس لضمان سرعة الوصول والتحقق
        self.pending_requests = {} 
        self.audit_log = []        
        self.last_log_hash = "0"   

    def create_transaction(self, maker_name: str, amount: float, description: str) -> str:
        """ينشئ طلب عملية جديد بمعرف فريد كلياً وتقييم آلي للمخاطر."""
        if amount <= 0:
            raise ValueError("❌ خطأ: يجب أن يكون مبلغ العملية أكبر من صفر!")

        # حل مشكلة تكرار المعرفات باستخدام UUID4
        unique_id = uuid.uuid4().hex[:8].upper()
        tx_id = f"TXN-{datetime.now().strftime('%Y%m%d')}-{unique_id}"
        
        # تقييم المخاطر (تم تحسين الشرط ليكون أكثر مرونة)
        risk_level = "High (عالي)" if amount > 50000 else "Medium (متوسط)"
        
        transaction = {
            "Transaction_ID": tx_id,
            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Maker": maker_name.strip(),
            "Amount": amount,
            "Description": description.strip(),
            "Risk_Level": risk_level,
            "Status": "Pending Approval"
        }
        
        self.pending_requests[tx_id] = transaction
        print(f"\n[+] [Maker] الموظف ({maker_name}) أنشأ عملية برقم: {tx_id}")
        print(f"    ⚠️ تقييم المخاطر التلقائي: {risk_level}")
        return tx_id

    def verify_transaction(self, tx_id: str, checker_name: str, action: str) -> bool:
        """يراجع العملية ويمنع الصانع من الاعتماد تحقيقاً لمبدأ الفصل بين الواجبات (SoD)."""
        tx = self.pending_requests.get(tx_id)
        
        if not tx:
            print(f"❌ خطأ: رقم العملية {tx_id} غير موجود أو تمت معالجته مسبقاً!")
            return False

        # حماية أمنية صارمة: الفصل بين الواجبات (Segregation of Duties)
        if tx["Maker"].lower() == checker_name.strip().lower():
            print(f"🚨 [انتهاك أمني] لا يمكن للموظف ({checker_name}) اعتماد عملية قام بإنشائها بنفسه!")
            return False

        # توحيد التحقق من حالة القرار
        is_approved = action.strip().lower() in ['اعتماد', 'approve', 'y', 'yes']
        
        tx["Status"] = "Approved & Executed" if is_approved else "Rejected"
        tx["Checker"] = checker_name.strip()
        tx["Verification_Time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # ترحيل السجل وتطهير القائمة المعلقة
        self._add_to_immutable_audit_log(tx)
        del self.pending_requests[tx_id]
        
        status_emoji = "✅" if is_approved else "❌"
        status_text = "باعتماد" if is_approved else "برفض"
        print(f"{status_emoji} [Checker] المشرف ({checker_name}) قام {status_text} العملية {tx_id} .")
        return True

    def _add_to_immutable_audit_log(self, transaction: dict):
        """يربط السجل الحالي بالسجل السابق عبر تشفير SHA-256 لمنع التلاعب (Blockchain Concept)."""
        # استخدام ensure_ascii=False للحفاظ على النصوص العربية داخل الـ JSON وبثبات الترتيب
        tx_string = json.dumps(transaction, sort_keys=True, ensure_ascii=False)
        
        # بناء الكتلة المشفرة
        combined_data = f"{tx_string}{self.last_log_hash}"
        current_hash = hashlib.sha256(combined_data.encode('utf-8')).hexdigest()
        
        audit_entry = {
            "Data": transaction,
            "Previous_Hash": self.last_log_hash,
            "Current_Hash": current_hash
        }
        
        self.audit_log.append(audit_entry)
        self.last_log_hash = current_hash 

    def display_audit_trail(self):
        """يعرض سجل التدقيق الأمني لغايات الامتثال الرقابي وضمان سلامة البيانات."""
        print("\n" + "="*25 + " 📊 سجل التدقيق الأمني (Audit Trail) " + "="*25)
        for entry in self.audit_log:
            data = entry['Data']
            print(f"ID: {data['Transaction_ID']} | الحالة: {data['Status']}")
            print(f"الصانع: {data['Maker']} | المدقق: {data.get('Checker', 'N/A')}")
            print(f"المبلغ: {data['Amount']} | المخاطر: {data['Risk_Level']}")
            print(f"🔗 الـ Hash الحالي : {entry['Current_Hash']}")
            print(f"⛓️ الـ Hash السابق: {entry['Previous_Hash']}")
            print("-" * 86)

if __name__ == "__main__":
    print(" 🛡️ RiskGuard: Maker-Checker & Immutable Audit System 🛡️ ")
    system = RiskGuardSystem()

    # 1. إنشاء عمليات
    tx1 = system.create_transaction(maker_name="أحمد", amount=75000, description="شراء خوادم جديدة")
    tx2 = system.create_transaction(maker_name="سارة", amount=12000, description="تجديد تراخيص")

    # 2. فحص محاولة اختراق السياسات الأمنية
    print("\n--- محاولة فحص اختراق الضوابط الأمنية ---")
    system.verify_transaction(tx_id=tx1, checker_name="أحمد", action="اعتماد")

    # 3. معالجة صحيحة (الفصل بين الواجبات)
    print("\n--- معالجة العمليات بشكل صحيح ---")
    system.verify_transaction(tx_id=tx1, checker_name="ريناد", action="approve") 
    system.verify_transaction(tx_id=tx2, checker_name="أحمد", action="اعتماد")  

    # 4. عرض سجل الامتثال
    system.display_audit_trail()


 🛡️ RiskGuard: Maker-Checker & Immutable Audit System 🛡️ 

[+] [Maker] الموظف (أحمد) أنشأ عملية برقم: TXN-20260615-F7756BCB
    ⚠️ تقييم المخاطر التلقائي: High (عالي)

[+] [Maker] الموظف (سارة) أنشأ عملية برقم: TXN-20260615-3459B544
    ⚠️ تقييم المخاطر التلقائي: Medium (متوسط)

--- محاولة فحص اختراق الضوابط الأمنية ---
🚨 [انتهاك أمني] لا يمكن للموظف (أحمد) اعتماد عملية قام بإنشائها بنفسه!

--- معالجة العمليات بشكل صحيح ---
✅ [Checker] المشرف (ريناد) قام باعتماد العملية TXN-20260615-F7756BCB .
✅ [Checker] المشرف (أحمد) قام باعتماد العملية TXN-20260615-3459B544 .

========================= 📊 سجل التدقيق الأمني (Audit Trail) =========================
ID: TXN-20260615-F7756BCB | الحالة: Approved & Executed
الصانع: أحمد | المدقق: ريناد
المبلغ: 75000 | المخاطر: High (عالي)
🔗 الـ Hash الحالي : 9c2eea651363e0c78435d59d6656fad190d8d289fae40bff2a27e8c27b322197
⛓️ الـ Hash السابق: 0
--------------------------------------------------------------------------------------
ID: TXN-20260615-3459B544 |